In [ ]:
%load_ext blackcellmagic 
# %black -l 120
%load_ext autoreload
%autoreload 2

In [ ]:
from slimdqn.algorithms.gidqnshared import GiDQNShared
from slimdqn.algorithms.idqnshared import iDQNShared
from slimdqn.algorithms.dqnrcshared import DQNRCShared
from slimdqn.algorithms.dqn import DQN

import jax
import jax.numpy as jnp
from tests.utils import Generator


def count_params(params):
	return sum(x.size for x in jax.tree.leaves(params))


def count_flops(q, has_target_params=False):
	best_action_compiled = jax.jit(q.best_action).lower(q.params, sample_generator.state(jax.random.PRNGKey(0))).compile()
	if not has_target_params:
		learn_on_batch_compiled = jax.jit(q.learn_on_batch).lower(q.params, q.optimizer_state, sample_generator.samples(jax.random.PRNGKey(0)), jnp.ones(32)).compile()
	else:
		learn_on_batch_compiled = jax.jit(q.learn_on_batch).lower(q.params, q.target_params, q.optimizer_state, sample_generator.samples(jax.random.PRNGKey(0)), jnp.ones(32)).compile()

	return best_action_compiled, learn_on_batch_compiled

sample_generator = Generator(32, (84, 84, 4), 10) 

architectures = ["cnn", "impala"]
feature_list = [[32, 64, 64, 512], [16, 32, 32, 512]]
metrics = {}
metrics["flops"] = {}
metrics["num_params"] = {}

for idx, architecture in enumerate(architectures):
	features = feature_list[idx]
	gap = (idx == 1)
	print(f"--- DQN---")
	q_dqn = DQN(jax.random.PRNGKey(0), (84, 84, 4), 10, features, architecture, False, gap, 6.25e-5, 0.99, 1, 0.25, 8000)
	metrics["num_params"][f"dqn_{architecture}"] = count_params(q_dqn.params) + count_params(q_dqn.target_params)
	q_dqn_best_action_compiled, q_dqn_learn_on_batch_compiled = count_flops(q_dqn, has_target_params=True)
	metrics["flops"][f"dqn_{architecture}"] = q_dqn_learn_on_batch_compiled.cost_analysis()[0]["flops"]
	print("DQN with vmap", metrics["num_params"][f"dqn_{architecture}"])
	print("DQN FLOPs best action: ", q_dqn_best_action_compiled.cost_analysis()[0]["flops"])
	print("DQN FLOPs to learn on a batch: ", metrics["flops"][f"dqn_{architecture}"], "\n")

	print(f"--- DQNRC ---")
	q_qrc = DQNRCShared(jax.random.PRNGKey(0), (84, 84, 4), 10, features, architecture, False, gap, True, 6.25e-5, 0.99, 1, 0.25, 8000, 1)
	metrics["num_params"][f"qrc_{architecture}"] = count_params(q_qrc.params)
	q_qrc_best_action_compiled, q_qrc_learn_on_batch_compiled = count_flops(q_qrc, has_target_params=False)
	metrics["flops"][f"qrc_{architecture}"] = q_qrc_learn_on_batch_compiled.cost_analysis()[0]["flops"]
	print("DQNRC with linear Heads", metrics["num_params"][f"qrc_{architecture}"])
	print("DQNRC FLOPs best action: ", q_qrc_best_action_compiled.cost_analysis()[0]["flops"])
	print("DQNRC FLOPs to learn on a batch: ", metrics["flops"][f"qrc_{architecture}"], "\n")

	print(f"--- i-DQN ---")
	q_idqn = iDQNShared(jax.random.PRNGKey(0), (84, 84, 4), 10, 5, features, architecture, False, gap, True, 6.25e-5, 0.99, 1, 0.25, 8000)
	metrics["num_params"][f"idqn_{architecture}"] = count_params(q_idqn.params) + count_params(q_idqn.target_params)
	q_idqn_best_action_compiled, q_idqn_learn_on_batch_compiled = count_flops(q_idqn, has_target_params=True)
	metrics["flops"][f"idqn_{architecture}"] = q_idqn_learn_on_batch_compiled.cost_analysis()[0]["flops"]
	print("iDQN with linear Heads", metrics["num_params"][f"idqn_{architecture}"])
	print("Linear i-DQN FLOPs best action: ", q_idqn_best_action_compiled.cost_analysis()[0]["flops"])
	print("Linear i-DQN FLOPs to learn on a batch: ", metrics["flops"][f"idqn_{architecture}"], "\n")

	print(f"--- Gi-DQN---")
	q_gidqn = GiDQNShared(jax.random.PRNGKey(0), (84, 84, 4), 10, 5, features, architecture, False, gap, True, 6.25e-5, 0.99, 1, 0.25, True, 8000, 1)
	metrics["num_params"][f"gidqn_{architecture}"] = count_params(q_gidqn.params) + count_params(q_gidqn.target_params)
	q_gidqn_best_action_compiled, q_gidqn_learn_on_batch_compiled = count_flops(q_gidqn, has_target_params=True)
	metrics["flops"][f"gidqn_{architecture}"] = q_gidqn_learn_on_batch_compiled.cost_analysis()[0]["flops"]
	print("GiDQN with linear Heads", metrics["num_params"][f"gidqn_{architecture}"])
	print("Linear FLOPs best action: ", q_gidqn_best_action_compiled.cost_analysis()[0]["flops"])
	print("Linear FLOPs to learn on a batch: ", metrics["flops"][f"gidqn_{architecture}"], "\n")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

AVAILABLE_COLORS = {
    "black": "#000000",
    "blue": "#1F77B4",
    "light_blue": "#AEC7E8",
    "orange": "#FF7F0E",
    "light_orange": "#FFBB78",
    "green": "#2CA02C",
    "light_green": "#98DF8A",
    "red": "#D1797A",
    "light_red": "#FF9896",
    "purple": "#9467BD",
    "light_purple": "#C5B0D5",
    "brown": "#8C564B",
    "light_brown": "#C49C94",
    "pink": "#E377C2",
    "light_pink": "#F7B6D2",
    "grey": "#7F7F7F",
    "light_grey": "#C7C7C7",
    "yellow": "#DEDE00",
    "light_yellow": "#F0E886",
    "cyan": "#17BECF",
    "light_cyan": "#9EDAE5",
}

plt.rc("font", family="STIXGeneral", serif="Times New Roman", size=10)
plt.rc("mathtext", fontset="stix")
plt.rcParams.update({
    "axes.linewidth": 0.4,

    "lines.linewidth": 1.5,

    "xtick.major.width": 0.45,
    "ytick.major.width": 0.45,
    "xtick.major.size": 1.8,
    "ytick.major.size": 1.8,

    "grid.linewidth": 0.45,
})

rt_cnn = [16.6, 23.3, 21.66, 25]
rt_impala = [50, 60, 58.33, 65]

def color(name):
    if name.split("_")[0] == "dqn":
        return "black"
    elif name.split("_")[0] == "qrc":
        return "purple"
    elif name.split("_")[0] == "idqn":
        return "orange"
    elif name.split("_")[0] == "gidqn":
        return "blue"

algo_names = ["dqn", "qrc", "idqn", "gidqn"]
fig, axes = plt.subplots(1, 3, figsize=(5.5, 1.5))
fig.subplots_adjust(
    left=0.04, right=0.98,
    bottom=0.2, top=0.87,
)

for idx_name, name in enumerate(algo_names):
    bar = axes[0].bar(idx_name, metrics["flops"][f"{name}_cnn"], hatch="\\\\", linewidth=0.0,edgecolor="white", color=AVAILABLE_COLORS[color(name)], zorder=2)
    axes[0].hlines(metrics["flops"][f"{name}_cnn"], idx_name - bar[0].get_width() / 2, idx_name + bar[0].get_width() / 2, color="white", zorder=4, linewidth=1)
    bar = axes[0].bar(idx_name, metrics["flops"][f"{name}_impala"], color=AVAILABLE_COLORS[color(name)], zorder=1)

for idx_name, name in enumerate(algo_names):
    bar = axes[1].bar(idx_name, rt_cnn[idx_name],edgecolor="white", hatch="\\\\", linewidth=0.0, color=AVAILABLE_COLORS[color(name)], zorder=2)    
    axes[1].hlines(rt_cnn[idx_name], idx_name - bar[0].get_width() / 2, idx_name + bar[0].get_width() / 2, color="white", zorder=4, linewidth=1)

    axes[1].bar(idx_name, rt_impala[idx_name], color=AVAILABLE_COLORS[color(name)], zorder=1)

for idx_name, name in enumerate(algo_names):
    bar = axes[2].bar(idx_name, metrics["num_params"][f"{name}_cnn"], edgecolor="white", hatch="\\\\",linewidth=0.0, color=AVAILABLE_COLORS[color(name)], zorder=1)
    axes[2].hlines(metrics["num_params"][f"{name}_impala"], idx_name - bar[0].get_width() / 2, idx_name + bar[0].get_width() / 2, color="white", zorder=4, linewidth=1)

    bar = axes[2].bar(idx_name, metrics["num_params"][f"{name}_impala"], color=AVAILABLE_COLORS[color(name)], zorder=2)

for ax in axes:
    ax.grid(visible=True)
    ax.set_xticks([])
    ax.set_xticklabels([])
    ax.set_axisbelow(True)

axes[2].ticklabel_format(axis='y', style="sci", useMathText=False)
axes[0].ticklabel_format(axis='y', style="sci", useMathText=False)
axes[2].set_yticklabels([])
axes[2].set_yscale("log")
axes[2].set_yticks([1e6, 1e7])
axes[2].set_yticklabels([1, 10])
axes[0].set_yticks([0, 1e10, 2e10])
axes[0].set_yticklabels([0, 1, 2])
axes[2].set_title(r"Number of Parameters ($\times 10^6$)", fontsize=10, pad=3)
axes[0].set_title(r"FLOPs ($\times 10^{10}$)", fontsize=10, pad=3)
axes[1].set_title("Runtime (in hours)", fontsize=10, pad=3)


In [ ]:
from matplotlib.patches import Patch
selected_legends = ["DQN", "QRC", "i-DQN", "Gi-DQN"]
selected_lines = [
    Patch(edgecolor="white", facecolor=AVAILABLE_COLORS["black"], linestyle="solid"),
    Patch(edgecolor="white", facecolor=AVAILABLE_COLORS["purple"]),
    Patch(edgecolor="white", facecolor=AVAILABLE_COLORS["orange"]),
    Patch(edgecolor="white", facecolor=AVAILABLE_COLORS["blue"]),
]
legend_algo = fig.legend(
    selected_lines,
    selected_legends,
    ncols=4,
    frameon=False,
    loc="center",
    bbox_to_anchor=(0.7, 0.07),
    columnspacing=1.5,
    handlelength=1,
    handletextpad=0.5,
    labelspacing=0.2,
    fontsize=11,
)
legend_algo.zorder = 0

selected_legends = ["CNN", "IMPALA+GAP"]
selected_lines = [
    Patch(edgecolor="white", facecolor=AVAILABLE_COLORS["light_grey"], hatch='\\\\\\', linewidth=0.5),
    Patch(edgecolor="white", facecolor=AVAILABLE_COLORS["light_grey"],linewidth=0.5),
]
legend_architecture = fig.legend(
    selected_lines,
    selected_legends,
    fancybox=False,
    ncols=2,
    frameon=True,
    framealpha=1,
    facecolor=AVAILABLE_COLORS["light_grey"],
    loc="center",
    bbox_to_anchor=(0.2, 0.07),
    handlelength=1,
    columnspacing=1,
    borderpad=0.05,
    handletextpad=0.4,
    labelspacing=0.0,
    fontsize=11,
)
fig.legends = [legend_algo, legend_architecture]

fig.savefig("analysis.pdf", pad_inches=0)
fig